In [1]:
import sys
import os

# Add the project root to the Python path
project_root = '/home/campus.ncl.ac.uk/c4071391/Projects/SLAM3R'
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f"Added {project_root} to sys.path")
# sys.path.append(project_root + 'slam3r')

print(sys.path)


Added /home/campus.ncl.ac.uk/c4071391/Projects/SLAM3R to sys.path
['/home/campus.ncl.ac.uk/c4071391/Projects/SLAM3R', '/home/campus.ncl.ac.uk/c4071391/miniconda3/envs/slam3r/lib/python311.zip', '/home/campus.ncl.ac.uk/c4071391/miniconda3/envs/slam3r/lib/python3.11', '/home/campus.ncl.ac.uk/c4071391/miniconda3/envs/slam3r/lib/python3.11/lib-dynload', '', '/home/campus.ncl.ac.uk/c4071391/miniconda3/envs/slam3r/lib/python3.11/site-packages']


In [2]:
import torch
import slam3r.pos_embed.curope.curope as curope_kernels

def rope_2d_pytorch(tokens, positions, base, F0):
    # tokens: (B, N, H, 4D), positions: (B, N, 2)
    B, N, H, Dtot = tokens.shape
    D = Dtot // 4
    device = tokens.device
    dtype = tokens.dtype

    d = torch.arange(D, device=device, dtype=dtype)  # (D,)
    base = torch.as_tensor(float(base), device=device, dtype=dtype)
    base_pow = base ** (d / float(D))               # (D,)

    tokens_view = tokens.view(B, N, H, 4, D)
    pos = positions.to(dtype)

    for axis in range(2):  # 0 -> y, 1 -> x
        pos_axis = pos[..., axis]                   # (B, N)
        inv = F0 * pos_axis[..., None] / base_pow   # (B, N, D)
        cos = torch.cos(inv)[..., None, :]          # (B, N, 1, D)
        sin = torch.sin(inv)[..., None, :]          # (B, N, 1, D)

        u = tokens_view[:, :, :, 0 + 2*axis, :]     # (B, N, H, D)
        v = tokens_view[:, :, :, 1 + 2*axis, :]

        u_new = u * cos - v * sin
        v_new = v * cos + u * sin

        tokens_view[:, :, :, 0 + 2*axis, :] = u_new
        tokens_view[:, :, :, 1 + 2*axis, :] = v_new

    # in-place, like the C++ op
    return tokens

# override the extension op
curope_kernels.rope_2d = rope_2d_pytorch


In [3]:
# !pip install netron
import torch
import onnx
import netron
import collections
from slam3r.models import Image2PointsModel, Local2WorldModel
import slam3r.blocks.multiview_blocks as mv

# monkey patch to not use xformers, not traceable by onnx export
mv.XFORMERS_AVALIABLE = False

import tqdm as tqdm_lib
from slam3r.utils import device as slam_device
import slam3r.models as slam_models

class DummyNvtxRange:
    def __init__(self, *args, **kwargs):
        pass
    def __enter__(self):
        return None
    def __exit__(self, exc_type, exc, tb):
        return False

# Disable NVTX ranges for export
slam_device.MyNvtxRange = DummyNvtxRange
slam_models.MyNvtxRange = DummyNvtxRange

# Disable tqdm progress bars for export
class DummyTqdm:
    def __init__(self, iterable=None, *args, **kwargs):
        self.iterable = iterable
    def __iter__(self):
        return iter(self.iterable)
    def __enter__(self):
        return self
    def __exit__(self, exc_type, exc, tb):
        return False
    def update(self, *args, **kwargs):
        pass

tqdm_lib.tqdm = DummyTqdm      # global tqdm
slam_models.tqdm = DummyTqdm   # tqdm imported inside slam3r.models

#  is not a standard PyTorch model, so we need its definition.
# Assuming the DROID model class is defined in a 'droid_net.py' file
# as is common in DROID-SLAM implementations.

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# --- 1. Load the PyTorch Model ---
i2p_model = Image2PointsModel.from_pretrained('siyan824/slam3r_i2p')
i2p_model.to(device)
i2p_model.eval()

i2p_num_params = sum(p.numel() for p in i2p_model.parameters())
print(f"i2p parameters: {i2p_num_params} (~{i2p_num_params/1e6:.1f}M)")

l2w_model = Local2WorldModel.from_pretrained('siyan824/slam3r_l2w')
l2w_model.to(device)
l2w_model.eval()

l2w_num_params = sum(p.numel() for p in l2w_model.parameters())
print(f"l2w parameters: {l2w_num_params} (~{l2w_num_params/1e6:.1f}M)")

onnx_i2p_model_path = 'i2p.onnx'
onnx_l2w_model_path = 'l2w.onnx'



i2p parameters: 532934913 (~532.9M)
l2w parameters: 230624000 (~230.6M)


In [6]:
# i2p 
# parser.add_argument('--i2p_model', type=str, default="Image2PointsModel(pos_embed='RoPE100', img_size=(224, 224), head_type='linear', output_mode='pts3d', depth_mode=('exp', -inf, inf), conf_mode=('exp', 1, inf), \


#  b, n, c1, h1, w1 = x.shape
t = 2  # number of time steps (must be >= 2 for multiview)
B = 1  # batch size
C = 3  # number of channels
H = 224  # height
W = 224  # width

# need to provide:
# images
dummy_images = torch.randn(B, t, C, H, W).to(device)

class I2PWrapper(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
        # The model expects a ref_id, let's fix it to 0
        self.ref_id = 0

    def forward(self, images):
        # The model's forward pass expects a list of view dictionaries.
        # Let's construct it from the input tensor.
        # images shape: (B, T, C, H, W)
        B, T, C, H, W = images.shape
        
        views = []
        for i in range(T):
            view = {
                'img': images[:, i],
                'true_shape': torch.tensor([[H, W]], device=images.device).repeat(B, 1)
            }
            views.append(view)
        
        # The model returns a list of dictionaries. For ONNX export,
        # we need to return tensors. Let's return the point cloud
        # of the first view.
        output = self.model(views, ref_id=self.ref_id)
        return output[0]['pts3d']

# Wrap the model
i2p_model_wrapped = I2PWrapper(i2p_model).to(device)
i2p_model_wrapped.eval()

from torch.onnx import dynamo_export

print("Starting ONNX export (dynamo)...")

onnx_i2p_model_path = "i2p.onnx"

ep = dynamo_export(
    i2p_model_wrapped,
    dummy_images,   # (B, T, C, H, W)
)
ep.save(onnx_i2p_model_path)

print(f"Model has been converted to ONNX and saved at {onnx_i2p_model_path}")


Starting ONNX export (dynamo)...


/home/campus.ncl.ac.uk/c4071391/miniconda3/envs/slam3r/lib/python3.11/site-packages/torch/onnx/_internal/_exporter_legacy.py:116: UserWarning: torch.onnx.dynamo_export only implements opset version 18 for now. If you need to use a different opset version, please register them with register_custom_op.
  warnings.warn(


OnnxExporterError: Failed to export the model to ONNX. Generating SARIF report at 'report_dynamo_export.sarif'. SARIF is a standard format for the output of static analysis tools. SARIF logs can be loaded in VS Code SARIF viewer extension, or SARIF web viewer (https://microsoft.github.io/sarif-web-component/). Please report a bug on PyTorch Github: https://github.com/pytorch/pytorch/issues

In [ ]:

# --- 3. Verify and Visualize the ONNX model ---
# Load the ONNX model
onnx_model = onnx.load(onnx_i2p_model_path)

# Check that the model is well-formed
onnx.checker.check_model(onnx_model)

print("ONNX model check passed.")
print("Starting Netron visualization server...")

# Visualize the model using Netron
# This will start a web server and open the model in your browser.
netron.start(onnx_i2p_model_path)

Starting ONNX export...


/home/campus.ncl.ac.uk/c4071391/Projects/SLAM3R/slam3r/models.py:469: TracerWarning: Using len to get tensor shape might cause the trace to be incorrect. Recommended usage would be tensor.shape[0]. Passing a tensor of different shape might lead to errors or silently give incorrect results.
  assert ref_id < len(views) and ref_id >= 0
/home/campus.ncl.ac.uk/c4071391/Projects/SLAM3R/slam3r/models.py:469: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  assert ref_id < len(views) and ref_id >= 0
/home/campus.ncl.ac.uk/c4071391/Projects/SLAM3R/slam3r/models.py:470: TracerWarning: Using len to get tensor shape might cause the trace to be incorrect. Recommended usage would be tensor.shape[0]. Passing a tensor of different shape might lead to errors or silently give incorrect re

RuntimeError: Tensor.__contains__ only supports Tensor or scalar, but you passed in a <class 'str'>.